# 🏆 3D-FUTURE Automated Evaluation Notebook for TRELLIS Pipeline

Notebook này được thiết kế chuyên biệt để đánh giá tự động chất lượng hình học 3D của pipeline **TRELLIS** kết hợp cùng **Grounded-SAM2** trên bộ dữ liệu **3D-FUTURE** (Alibaba Tmall).

### Quy trình đánh giá:
1. Tải bộ dữ liệu **3D-FUTURE** (mô hình CAD gốc và ảnh chụp thực tế).
2. Chọn ngẫu nhiên 15 mẫu nội thất đại diện cho các nhóm: Giường, Ghế, Sofa, Bàn, Tủ.
3. Chạy toàn bộ pipeline **Grounded-SAM2** (nhận diện + tách nền) -> **TRELLIS** (sinh mô hình 3D) trên 15 ảnh mẫu.
4. Căn chỉnh tọa độ bằng thuật toán **SVD-ICP** và tính toán **Chamfer Distance (L1/L2)** & **F-Score** so sánh trực tiếp với mô hình CAD chuẩn của thiết kế viên.

---
## 🛠️ Bước 1: Đồng bộ và cài đặt Môi trường
Cài đặt các thư viện cần thiết trong môi trường ảo `/opt/venv310` và chuẩn bị các checkpoint trọng số.

In [ ]:
# 1. Đảm bảo setuptools/pkg_resources được cài đặt trước
!/opt/venv310/bin/pip install -q setuptools>=69.0

# 2. Cài đặt các thư viện bổ trợ cho DINO và SAM2 nếu chưa có
!/opt/venv310/bin/pip install -q groundingdino-py timm supervision addict yapf trimesh scipy

# 3. Tải checkpoints GroundingDINO
import os
os.makedirs("/kaggle/working/groundingdino_ckpt", exist_ok=True)
if not os.path.exists("/kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth"):
    print("⏳ Đang tải checkpoint GroundingDINO...")
    !wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth -O /kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth
    !wget -q https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/blueprint/GroundingDINO_SwinT_OGC.py -O /kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py
    print("✓ GroundingDINO weights ready!")

# 4. Tải checkpoints SAM2
os.makedirs("/kaggle/working/sam2_ckpt", exist_ok=True)
if not os.path.exists("/kaggle/working/sam2_ckpt/sam2_hiera_small.pt"):
    print("⏳ Đang tải checkpoint SAM2...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt -O /kaggle/working/sam2_ckpt/sam2_hiera_small.pt
    print("✓ SAM2 weights ready!")

---
## 📦 Bước 2: Tải Dataset 3D-FUTURE
Sử dụng thư viện `kagglehub` để tải dataset `3d-future-model` chính thức.

In [ ]:
import kagglehub

print("⏳ Đang tải bộ dữ liệu 3D-FUTURE (3D Furniture Dataset)...")
future_path = kagglehub.dataset_download("tobetheonly/3d-future-model")
print('✓ Đã tải về và lưu tại:', future_path)

---
## 🎲 Bước 3: Lọc và chọn ngẫu nhiên 15 Mẫu nội thất
Lọc ra các mẫu hợp lệ thuộc các danh mục: Giường, Ghế, Sofa, Bàn, Tủ để chuẩn bị kiểm thử.

In [ ]:
import os, json, random

# Định nghĩa đường dẫn gốc (dataset của kagglehub thường lưu trong .cache/kagglehub hoặc tương đương)
FUTURE_ROOT = '/kaggle/input/datasets/tobetheonly/3d-future-model/3D-FUTURE-model'
if not os.path.exists(FUTURE_ROOT):
    # Fallback tìm kiếm đường dẫn kagglehub
    import glob
    paths = glob.glob("/root/.cache/kagglehub/datasets/tobetheonly/3d-future-model/**/3D-FUTURE-model", recursive=True)
    if paths:
        FUTURE_ROOT = paths[0]

model_info_path = os.path.join(FUTURE_ROOT, 'model_info.json')

if not os.path.exists(model_info_path):
    raise FileNotFoundError(f"Không tìm thấy file model_info.json tại: {FUTURE_ROOT}. Vui lòng kiểm tra lại dataset.")

with open(model_info_path, 'r', encoding='utf-8') as f:
    model_info = json.load(f)

target_super_categories = ['Bed', 'Chair', 'Sofa', 'Table', 'Cabinet/Shelf/Desk']
candidates = [m for m in model_info if m['super-category'] in target_super_categories]

random.seed(42)
samples = random.sample(candidates, min(15, len(candidates)))

eval_samples = []
for s in samples:
    model_id = s['model_id']
    model_dir = os.path.join(FUTURE_ROOT, model_id)
    gt_mesh_path = os.path.join(model_dir, 'raw_model.obj')
    image_path = os.path.join(model_dir, 'image.jpg')
    if os.path.exists(gt_mesh_path) and os.path.exists(image_path):
        eval_samples.append({
            'model_id': model_id,
            'super_category': s['super-category'],
            'category': s['category'],
            'gt_mesh_path': gt_mesh_path,
            'image_path': image_path,
        })

print(f'📦 Đã chọn {len(eval_samples)} mẫu nội thất hợp lệ để đánh giá tự động:')
for idx, e in enumerate(eval_samples):
    print(f"  [{idx+1}] {e['model_id']} ({e['super_category']} / {e['category']})")

# Ghi file cấu hình tạm
eval_config_path = "/tmp/eval_samples_meta.json"
with open(eval_config_path, "w") as f:
    json.dump(eval_samples, f)

---
## ⚙️ Bước 4: Chạy Tự động Batch Pipeline trên 15 Mẫu ảnh
Gọi GroundingDINO -> SAM2 -> TRELLIS trên từng ảnh của 3D-FUTURE.

In [ ]:
import subprocess, sys

print("⏳ Đang chuẩn bị kịch bản thực thi pipeline... ")

batch_pipeline_script = """
import sys, os, json, torch, numpy as np
from PIL import Image
import shutil
import scipy.ndimage as ndimage

sys.path.insert(0, "/opt/venv310/lib/python3.10/site-packages")
sys.modules['triton'] = None

os.environ["SPCONV_ALGO"]  = "native"
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPARSE_ATTN"]  = "xformers"
os.environ["MPLBACKEND"]   = "agg"

from groundingdino.util.inference import load_model, load_image, predict
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sys.path.insert(0, "/kaggle/working/TRELLIS")
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

# Nạp cấu hình các mẫu thử nghiệm
with open("/tmp/eval_samples_meta.json", "r") as f:
    eval_samples = json.load(f)

print("⏳ [PIPELINE] Đang tải các mô hình GroundingDINO, SAM2 và TRELLIS...")
DINO_CKPT = "/kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth"
DINO_CONFIG = "/kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py"
model_dino = load_model(DINO_CONFIG, DINO_CKPT)

SAM2_CKPT = "/kaggle/working/sam2_ckpt/sam2_hiera_small.pt"
sam2_model = build_sam2("sam2_hiera_s.yaml", SAM2_CKPT, device="cuda")
predictor = SAM2ImagePredictor(sam2_model)

pipeline = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
pipeline.to("cuda")

os.makedirs("/kaggle/working/outputs/trellis/eval_models", exist_ok=True)

print("\n🚀 Bắt đầu chạy Batch Pipeline...")
for idx, e in enumerate(eval_samples):
    mid = e['model_id']
    img_path = e['image_path']
    super_cat = e['super_category']
    out_mesh_path = f"/kaggle/working/outputs/trellis/eval_models/{mid}.glb"
    
    if os.path.exists(out_mesh_path):
        print(f"✓ [{idx+1}/{len(eval_samples)}] Model {mid} đã có sẵn, bỏ qua.")
        continue
        
    print(f"\n⏳ [{idx+1}/{len(eval_samples)}] Đang xử lý {mid} ({super_cat})...")
    
    label_map = {
        'Bed': 'bed',
        'Chair': 'chair',
        'Sofa': 'sofa',
        'Table': 'table',
        'Cabinet/Shelf/Desk': 'cabinet'
    }
    detector_label = label_map.get(super_cat, 'furniture')
    
    try:
        # A. Chạy GroundingDINO nhận diện box
        image_source, image_tensor = load_image(img_path)
        boxes, logits, phrases = predict(
            model=model_dino,
            image=image_tensor,
            caption=detector_label,
            box_threshold=0.30,
            text_threshold=0.25
        )
        
        H, W, _ = image_source.shape
        if len(boxes) == 0:
            print("    ⚠️ Không phát hiện vật thể, dùng box mặc định là toàn ảnh.")
            x1, y1, x2, y2 = 0, 0, W, H
        else:
            best_box_idx = torch.argmax(logits).item()
            cx, cy, bw, bh = boxes[best_box_idx].tolist()
            x1 = int((cx - bw/2) * W)
            y1 = int((cy - bh/2) * H)
            x2 = int((cx + bw/2) * W)
            y2 = int((cy + bh/2) * H)
            
        # B. Chạy SAM2 tách nền vật thể
        img_rgb = np.array(Image.open(img_path).convert("RGB"))
        predictor.set_image(img_rgb)
        
        input_box = np.array([[x1, y1, x2, y2]])
        cx_px = (x1 + x2) // 2
        cy_px = (y1 + y2) // 2
        point_coords = np.array([[cx_px, cy_px]])
        point_labels = np.array([1])
        
        masks, scores, _ = predictor.predict(
            point_coords=point_coords,
            point_labels=point_labels,
            box=input_box,
            multimask_output=False
        )
        
        closed_mask = ndimage.binary_closing(masks[0], structure=np.ones((7, 7)))
        filled_mask = ndimage.binary_fill_holes(closed_mask)
        
        alpha_array = (filled_mask * 255).astype(np.uint8)
        alpha_smooth = ndimage.gaussian_filter(alpha_array.astype(float), sigma=1.2)
        alpha_smooth = np.clip(alpha_smooth, 0, 255).astype(np.uint8)
        
        img_rgba = Image.fromarray(img_rgb).convert("RGBA")
        alpha = Image.fromarray(alpha_smooth)
        img_rgba.putalpha(alpha)
        
        # Cắt và crop có padding
        PAD = 15
        cx1 = max(0, x1 - PAD)
        cy1 = max(0, y1 - PAD)
        cx2 = min(W, x2 + PAD)
        cy2 = min(H, y2 + PAD)
        crop = img_rgba.crop((cx1, cy1, cx2, cy2))
        
        tmp_crop = f"/tmp/crop_eval_{mid}.png"
        crop.save(tmp_crop)
        
        # C. Chạy TRELLIS Image-to-3D
        img_trellis = Image.open(tmp_crop).convert("RGB")
        image_trellis = pipeline.preprocess_image(img_trellis)
        
        outputs = pipeline.run(
            image_trellis,
            seed=42,
            formats=["gaussian", "mesh"],
            preprocess_image=False,
            sparse_structure_sampler_params={"steps": 12, "cfg_strength": 7.5},
            slat_sampler_params={"steps": 12, "cfg_strength": 3.0},
        )
        
        glb = postprocessing_utils.to_glb(
            outputs["gaussian"][0], outputs["mesh"][0],
            simplify=0.95, texture_size=1024, verbose=False
        )
        glb.export(out_mesh_path)
        print(f"    ✓ Lưu GLB thành công tại: {out_mesh_path}")
    except Exception as ex:
        print(f"    ❌ Lỗi trong quá trình xử lý: {ex}")
    torch.cuda.empty_cache()
"""

with open("/tmp/run_batch_future_eval.py", "w") as f:
    f.write(batch_pipeline_script)

# Chạy tiến trình con trong môi trường venv310 độc lập tránh xung đột thư viện
print("⏳ Đang thực thi batch pipeline trên GPU (3-4 phút)...\n")
r = subprocess.run(["/opt/venv310/bin/python" if os.path.exists("/opt/venv310/bin/python") else sys.executable, "/tmp/run_batch_future_eval.py"])
if r.returncode == 0:
    print("\n✅ Hoàn tất tạo mô hình 3D cho toàn bộ 15 mẫu dữ liệu!")
else:
    print("\n❌ Có lỗi xảy ra trong quá trình sinh mô hình 3D.")

---
## 🏆 Bước 5: Chạy Đánh giá Hình học 3D (Chamfer Distance & F-Score)
Căn chỉnh bằng SVD-ICP và đo đạc sai số hình học của các mô hình đã tạo.

In [ ]:
import os, json
import numpy as np
import trimesh
from scipy.spatial import KDTree

# ── HÀM CHUẨN HÓA MESH ──
def normalize_mesh(mesh):
    centroid = mesh.bounding_box.centroid
    mesh.vertices -= centroid
    extents = mesh.extents
    max_extent = np.max(extents)
    if max_extent > 0:
        mesh.vertices /= max_extent
    return mesh

# ── THUẬT TOÁN SVD-ICP ──
def icp_align(source_pts, target_pts, max_iterations=50, tolerance=1e-5):
    src = np.copy(source_pts)
    dst = np.copy(target_pts)
    t_accum = np.mean(dst, axis=0) - np.mean(src, axis=0)
    src = src + t_accum
    prev_error = 0
    for i in range(max_iterations):
        tree = KDTree(dst)
        distances, indices = tree.query(src)
        matched_dst = dst[indices]
        c_src = np.mean(src, axis=0)
        c_dst = np.mean(matched_dst, axis=0)
        H = (src - c_src).T @ (matched_dst - c_dst)
        U, S, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:
            Vt[2, :] *= -1
            R = Vt.T @ U.T
        t = c_dst - c_src @ R.T
        src = src @ R.T + t
        mean_error = np.mean(distances)
        if abs(mean_error - prev_error) < tolerance:
            break
        prev_error = mean_error
    return src

# ── TÍNH CHAMFER DISTANCE & F-SCORE ──
def evaluate_geometry(gen_mesh_path, gt_mesh_path, num_samples=10000, threshold=0.02):
    gen_mesh = trimesh.load(gen_mesh_path, force='mesh')
    gt_mesh = trimesh.load(gt_mesh_path, force='mesh')
    gen_mesh = normalize_mesh(gen_mesh)
    gt_mesh = normalize_mesh(gt_mesh)
    gen_pts, _ = trimesh.sample.sample_surface(gen_mesh, num_samples)
    gt_pts, _ = trimesh.sample.sample_surface(gt_mesh, num_samples)
    aligned_gen_pts = icp_align(gen_pts, gt_pts)
    tree_gen = KDTree(aligned_gen_pts)
    tree_gt = KDTree(gt_pts)
    dist_gen_to_gt, _ = tree_gt.query(aligned_gen_pts)
    dist_gt_to_gen, _ = tree_gen.query(gt_pts)
    cd_l2 = np.mean(dist_gen_to_gt**2) + np.mean(dist_gt_to_gen**2)
    cd_l1 = np.mean(dist_gen_to_gt) + np.mean(dist_gt_to_gen)
    precision = np.mean(dist_gen_to_gt < threshold)
    recall = np.mean(dist_gt_to_gen < threshold)
    f_score = (2.0 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"cd_l1": cd_l1, "cd_l2": cd_l2, "precision": precision, "recall": recall, "f_score": f_score}

# NẠP CẤU HÌNH
with open("/tmp/eval_samples_meta.json", "r") as f:
    eval_samples = json.load(f)

print("=== 🏁 KHỞI CHẠY ĐÁNH GIÁ ĐỘ TƯƠNG ĐỒNG HÌNH HỌC 3D (TRELLIS) ===")
all_results = []
for e in eval_samples:
    mid = e['model_id']
    gen_path = f"/kaggle/working/outputs/trellis/eval_models/{mid}.glb"

    if not os.path.exists(gen_path):
        print(f"⚠️ Bỏ qua {mid} ({e['super_category']}): chưa có mesh TRELLIS sinh ra")
        continue

    try:
        res = evaluate_geometry(gen_path, e['gt_mesh_path'], num_samples=10000, threshold=0.02)
        res['model_id'] = mid
        res['super_category'] = e['super_category']
        all_results.append(res)
        print(f"✓ {mid} ({e['super_category']}): CD_L2={res['cd_l2']:.5f}  F-Score={res['f_score']*100:.2f}%")
    except Exception as ex:
        print(f"❌ Lỗi khi xử lý {mid}: {ex}")

if all_results:
    print("\n=======================================================")
    print(f"📊 KẾT QUẢ TRUNG BÌNH TRÊN {len(all_results)} OBJECT")
    print("=======================================================")
    cds_l1 = [r['cd_l1'] for r in all_results]
    cds_l2 = [r['cd_l2'] for r in all_results]
    fs = [r['f_score'] for r in all_results]
    print(f"• Chamfer Distance (L1): {np.mean(cds_l1):.6f} ± {np.std(cds_l1):.6f}")
    print(f"• Chamfer Distance (L2): {np.mean(cds_l2):.6f} ± {np.std(cds_l2):.6f}")
    print(f"• F-Score @ 0.02:        {np.mean(fs)*100:.2f}% ± {np.std(fs)*100:.2f}%")
    print("=======================================================\n")
else:
    print("⚠️ Không tìm thấy kết quả đánh giá nào.")